In [1]:
import scipy.optimize as opt
import numpy as np
import obs_to_obs_seq_in as obsin
import moored_obs_generator as obsgen
import datetime as dt
import pydartdiags.obs_sequence.obs_sequence as obsq
import xarray as xr
import true_w_mean as wmean

In [2]:
casename = 'EEP_MITgcm185Lvgrid_Whitt2026hgrid'
path = '/glade/derecho/scratch/iranjan/archive/' + casename + '/ocn/hist/'
files = casename + '.mom6.h.z.2015-0*-*.nc'

In [ ]:
true_ds = xr.open_mfdataset(path+files,drop_variables=["average_DT",])
true_ds['uh'] = true_ds['umo']/1035.
true_ds['vh'] = true_ds['vmo']/1035. 
true_ds = true_ds.rename({'z_l':'zl','z_i':'zi'})
true_ds['w_est1'] = true_ds['vert_remap_h_tendency'].cumsum(dim='zl')
weekly_ds = wmean.weekly_mean_w_est_profile(ds, center_lat=0.5, center_lon=220.0, box_size=1)

Box: 12 x 12 grid cells
Lat: 0.042 to 0.958
Lon: 219.542 to 220.458


In [ ]:
def opt_loc(loc_list, true_w):
    obs_seq = obsin.create_obs_seq_in()

    # TO-DO: add as inputs? (dt.datetime objects)
    start_year = 2015
    start_month = 1
    start_date = 1
    start_time = 1
    end_year = 2015
    end_month = 1
    end_date = 3
    end_time = 23
    # adds obs to obs_seq
    for lat, lon in loc_list:
        obsgen.MooredObs(obs_seq,lon,lat,8,80,2,"height (m)",dt.datetime(start_year,start_month,start_date,start_time),dt.datetime(end_year,end_month,end_date,end_time),0.001,dt.timedelta(hours=3))
    # splits by time and writes to subdirectories, runs pmo, obs_seq_to_netcdf, saves nc files
    obsin.split_obs_seq_by_time(obs_seq, "/glade/derecho/scratch/iranjan/archive/eep-slice-test/split", "EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.")
    # this is where we call planefit!
    # w_est = compute_w_planefit(loclist)
    w_est = true_w + np.random.rand()
    return np.linalg.norm(true_w - w_est)

In [ ]:
loc_list = [(0, 219), (1, 219), (0, 221), (1, 221)]

In [ ]:
result = opt_loc(loc_list, 5)

In [ ]:
print(result)